In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "research":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "config" / "config.yaml").exists():
    PROJECT_ROOT = Path(r"C:\Users\vishn\Desktop\NLP\End-to-end-Text-Summarizer")

os.chdir(PROJECT_ROOT)
print(f"Project root: {Path.cwd()}")
print(f"Config exists: {(Path('config') / 'config.yaml').exists()}")
print(f"Params exists: {Path('params.yaml').exists()}")


In [ ]:
from pathlib import Path

Path.cwd()


In [ ]:
# Already moved to the project root in the first cell.


In [ ]:
from pathlib import Path

Path.cwd()


In [ ]:
from pathlib import Path

print(f"Working directory: {Path.cwd()}")
print(Path("config/config.yaml").exists())
print(Path("params.yaml").exists())


In [ ]:
from pathlib import Path

ROOT_DIR = Path(r"C:\Users\vishn\Desktop\NLP\End-to-end-Text-Summarizer")

CONFIG_FILE_PATH = ROOT_DIR / "config" / "config.yaml"
PARAMS_FILE_PATH = ROOT_DIR / "params.yaml"

In [ ]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    data_path: Path
    model_ckpt: str
    num_train_epochs: int
    warmup_steps: int
    per_device_train_batch_size: int
    weight_decay: float
    logging_steps: int
    eval_strategy: str
    eval_steps: int
    save_steps: float
    gradient_accumulation_steps: int


In [ ]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [ ]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.TrainingArguments
        eval_strategy = getattr(params, "eval_strategy", None) or getattr(params, "evaluation_strategy", "steps")
        save_steps = float(params.save_steps)

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_ckpt=config.model_ckpt,
            num_train_epochs=params.num_train_epochs,
            warmup_steps=params.warmup_steps,
            per_device_train_batch_size=params.per_device_train_batch_size,
            weight_decay=params.weight_decay,
            logging_steps=params.logging_steps,
            eval_strategy=eval_strategy,
            eval_steps=params.eval_steps,
            save_steps=save_steps,
            gradient_accumulation_steps=params.gradient_accumulation_steps
        )

        return model_trainer_config


In [ ]:
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForSeq2Seq
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_dataset, load_from_disk
import torch

In [ ]:
import inspect
import os


class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config
        os.environ["WANDB_DISABLED"] = "true"

    def _training_arguments(self) -> TrainingArguments:
        kwargs = {
            "output_dir": self.config.root_dir,
            "num_train_epochs": self.config.num_train_epochs,
            "warmup_steps": self.config.warmup_steps,
            "per_device_train_batch_size": self.config.per_device_train_batch_size,
            "per_device_eval_batch_size": self.config.per_device_train_batch_size,
            "weight_decay": self.config.weight_decay,
            "logging_steps": self.config.logging_steps,
            "eval_steps": self.config.eval_steps,
            "save_steps": self.config.save_steps,
            "gradient_accumulation_steps": self.config.gradient_accumulation_steps,
            "report_to": "none",
        }

        argument_names = inspect.signature(TrainingArguments.__init__).parameters
        if "eval_strategy" in argument_names:
            kwargs["eval_strategy"] = self.config.eval_strategy
        else:
            kwargs["evaluation_strategy"] = self.config.eval_strategy

        return TrainingArguments(**kwargs)

    def train(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained(self.config.model_ckpt)
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_ckpt).to(device)
        seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)

        dataset_samsum_pt = load_from_disk(self.config.data_path)
        train_split = "test" if "test" in dataset_samsum_pt else "train"
        eval_split = "validation" if "validation" in dataset_samsum_pt else train_split

        trainer = Trainer(
            model=model_pegasus,
            args=self._training_arguments(),
            tokenizer=tokenizer,
            data_collator=seq2seq_data_collator,
            train_dataset=dataset_samsum_pt[train_split],
            eval_dataset=dataset_samsum_pt[eval_split],
        )

        trainer.train()

        model_pegasus.save_pretrained(os.path.join(self.config.root_dir, "pegasus-samsum-model"))
        tokenizer.save_pretrained(os.path.join(self.config.root_dir, "tokenizer"))


In [ ]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer_config = ModelTrainer(config=model_trainer_config)
    model_trainer_config.train()
except Exception as e:
    raise e